> 🔑 **SOLUTIONS NOTEBOOK**: instructor / self-check copy. Try the problems in `../notebooks/` first!

# ⚖️ Lesson 06: Cross-Validation & Model Comparison

**Time:** ~3 hours  **Prerequisites:** Lessons 00–05

Everything so far has a quiet flaw. We trained on 80% of the data, tested on 20%, read off an RMSE to one decimal place (*"Model B: 107.8, beats Model A: 108.1!"*) and moved on.
**How much would those numbers change if we had picked a different 20%?** If the answer is "a lot", then our confident rankings were built on sand.

This lesson turns you from someone who *runs* models into someone who can **defend a model choice with evidence**. That's the entire "Expected Outcome" of Phase 6.

### 🎯 Learning goals
1. Quantify how noisy a single train/test split is.
2. Use **K-fold, stratified, repeated, group and time-aware** cross-validation, and know which one your problem needs.
3. Spot and prevent **data leakage** (with a demonstration that will make you wince).
4. Compare models with **paired fold scores**, the **corrected resampled t-test**, and the **one-standard-error rule**.
5. Use **nested cross-validation** to estimate performance when you also tune hyperparameters.
6. Choose metrics that match the business question.

In [1]:
import sys; sys.path.append("../utils")          # lets us import the course helpers
from course_utils import *                       # load_housing(), check(), set_style(), ...
import numpy as np, pandas as pd, matplotlib.pyplot as plt

set_style()
np.random.seed(SEED)
from scipy import stats
from sklearn.model_selection import (train_test_split, KFold, StratifiedKFold, RepeatedKFold, GroupKFold, cross_val_score,
                                     cross_validate, cross_val_predict, learning_curve, GridSearchCV)
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.compose import TransformedTargetRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.feature_selection import SelectKBest, f_classif
import xgboost as xgb
import seaborn as sns

X, y = housing_xy()                      # the full housing data (3,000 rows); ids removed
def rmse(a, b): return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

## 1 · The split lottery

Let's repeat the *same* experiment (fit HistGradientBoosting and XGBoost, score on a held-out 20%) with **40 different random splits**, and watch the numbers move.

In [2]:
rows = []
for seed in range(40):
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, random_state=seed)
    prep = make_preprocessor(Xa); A, B = prep.fit_transform(Xa), prep.transform(Xb)
    hgb = HistGradientBoostingRegressor(learning_rate=0.05, max_iter=300, max_depth=4, random_state=0).fit(A, ya)
    xg = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, subsample=0.8, random_state=0).fit(A, ya)
    rows.append({"seed": seed, "HistGB": rmse(yb, hgb.predict(B)), "XGBoost": rmse(yb, xg.predict(B))})
lottery = pd.DataFrame(rows)
lottery["XGB − HistGB"] = lottery["XGBoost"] - lottery["HistGB"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(lottery["HistGB"], bins=12, alpha=.7, label="HistGB"); axes[0].hist(lottery["XGBoost"], bins=12, alpha=.7, label="XGBoost")
axes[0].set_xlabel("test RMSE"); axes[0].set_title("Same model, 40 different test sets"); axes[0].legend()
axes[1].hist(lottery["XGB − HistGB"], bins=12, color=PALETTE["purple"], alpha=.8); axes[1].axvline(0, color="k")
axes[1].set_xlabel("XGBoost RMSE − HistGB RMSE (same split)"); axes[1].set_title("The *difference* on the same split")
plt.tight_layout(); plt.show()

print(lottery[["HistGB", "XGBoost"]].describe().loc[["mean", "std", "min", "max"]].round(1))
print(f"\nXGBoost beat HistGB on {(lottery['XGB − HistGB'] < 0).mean():.0%} of the splits; the std. dev. of the difference is only {lottery['XGB − HistGB'].std():.1f}")

      HistGB  XGBoost
mean   115.0    109.2
std     13.6     13.6
min     90.4     87.0
max    147.3    140.2

XGBoost beat HistGB on 100% of the splits; the std. dev. of the difference is only 2.7


### 🔎 Two lessons in one picture

1. **Absolute scores are noisy.** The *same model* scores anywhere from about 90 to 147 depending on which houses landed in the test set (std. dev. ≈ 13, i.e. ±12%). Quoting "RMSE = 108.2" to one decimal is **false precision**. A single split gives you a number, not a measurement.
2. **Differences on the same split are far more stable.** Both models face the *same* test houses, so the hard-to-predict mansions hurt both equally; the *difference* has a std. dev. of only ≈ 2.7. **Comparing models on identical folds ("paired" comparison) cancels most of the noise.** This idea powers the rest of the lesson.

## 2 · K-fold cross-validation

Instead of one split, **rotate** the roles: cut the data into $K$ folds; train on $K-1$, validate on the held-out fold; repeat $K$ times; average.

```
  fold 1:  [ VAL ][ train ][ train ][ train ][ train ]   -> score_1
  fold 2:  [ train ][ VAL ][ train ][ train ][ train ]   -> score_2
  fold 3:  [ train ][ train ][ VAL ][ train ][ train ]   -> score_3
  fold 4:  [ train ][ train ][ train ][ VAL ][ train ]   -> score_4
  fold 5:  [ train ][ train ][ train ][ train ][ VAL ]   -> score_5
                                                   CV estimate = mean(score_1..5)   (± spread)
```

**Every row is used for validation exactly once, and for training K−1 times.** Typical $K$: 5 or 10. Larger $K$ → less pessimistic estimate (more training data per fold) but more computation and (slightly) more variance between folds.

Let's see the mechanics on ten rows:

In [3]:
for i, (tr, va) in enumerate(KFold(n_splits=5, shuffle=True, random_state=0).split(np.arange(10)), 1):
    print(f"fold {i}: train rows {sorted(tr.tolist())}   validate rows {sorted(va.tolist())}")

fold 1: train rows [0, 1, 3, 4, 5, 6, 7, 9]   validate rows [2, 8]
fold 2: train rows [0, 1, 2, 3, 5, 6, 7, 8]   validate rows [4, 9]
fold 3: train rows [0, 2, 3, 4, 5, 7, 8, 9]   validate rows [1, 6]
fold 4: train rows [0, 1, 2, 4, 5, 6, 8, 9]   validate rows [3, 7]
fold 5: train rows [1, 2, 3, 4, 6, 7, 8, 9]   validate rows [0, 5]


### CV in one line: and the pipeline rule

`cross_val_score` fits a fresh copy of the model on each training fold and scores the validation fold.
The crucial habit: **put preprocessing inside a `Pipeline`**, so that imputation, scaling, encoding and feature selection are *re-fitted on each training fold only* (more on why in Section 4).

In [4]:
def housing_pipe(model):
    return make_pipeline(make_preprocessor(X, scale=True), model)

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
scores = -cross_val_score(housing_pipe(HistGradientBoostingRegressor(learning_rate=0.05, max_iter=300, max_depth=4, random_state=0)),
                          X, y, cv=kf, scoring="neg_root_mean_squared_error")      # sklearn maximises, so RMSE comes back negated
print("RMSE per fold:", scores.round(1))
print(f"CV estimate: {scores.mean():.1f} ± {scores.std(ddof=1):.1f}   (the ± is the fold-to-fold spread)")

RMSE per fold: [110.4 125.2 118.6 116.5 108.1]
CV estimate: 115.7 ± 6.8   (the ± is the fold-to-fold spread)


## 3 · Choosing the right splitter (one size does *not* fit all)

The splitter encodes **a claim about the future**: "new data will look like *this*". Choose the one that matches reality.

| splitter | use when | scikit-learn |
|---|---|---|
| **K-fold** (shuffled) | rows are independent and identically distributed | `KFold` |
| **Stratified K-fold** | classification, especially **imbalanced**: keeps class proportions in each fold | `StratifiedKFold` |
| **Repeated K-fold** | you want a more stable estimate or enough folds for statistical tests | `RepeatedKFold` |
| **Group K-fold** | rows come in **groups** (same patient, same customer, same neighbourhood) and you will predict for *new groups* | `GroupKFold` |
| **Time-series split** | the future must not leak into the past (forecasting) | `TimeSeriesSplit` or "forward chaining" |

### Stratification: why imbalance needs it
A tiny dataset with 5 positives out of 100. Plain shuffled K-fold can produce folds with *zero* positives, and a metric like AUC is undefined there:

In [5]:
y_small = np.array([1] * 5 + [0] * 95)
Xd = np.zeros((100, 1))
print("positives per validation fold")
print("  KFold (shuffled):", [int(y_small[va].sum()) for _, va in KFold(5, shuffle=True, random_state=1).split(Xd)])
print("  StratifiedKFold :", [int(y_small[va].sum()) for _, va in StratifiedKFold(5, shuffle=True, random_state=1).split(Xd, y_small)])

positives per validation fold
  KFold (shuffled): [0, 1, 2, 1, 1]
  StratifiedKFold : [1, 1, 1, 1, 1]


### Groups and time: validation that mimics deployment

We'll estimate the *same model's* error with three validation designs:

1. **Random K-fold**: "tomorrow's listings are like today's, in the same neighbourhoods and the same year."
2. **Group K-fold by neighbourhood**: "we're launching in *neighbourhoods we've never seen*."
3. **Forward chaining by year**: "we train on the past and predict next year" (train on years < Y, test on year Y).

🔮 **Predict:** which design will give the *highest* (most pessimistic) RMSE, and which the lowest?

In [6]:
def make_hgb(): return housing_pipe(HistGradientBoostingRegressor(learning_rate=0.05, max_iter=300, max_depth=4, random_state=0))

random_cv = -cross_val_score(make_hgb(), X, y, cv=KFold(5, shuffle=True, random_state=SEED), scoring="neg_root_mean_squared_error")
group_cv = -cross_val_score(make_hgb(), X, y, cv=GroupKFold(n_splits=5), groups=X["neighborhood"], scoring="neg_root_mean_squared_error")

forward = {}
for year in [2021, 2022, 2023, 2024]:
    tr, te = (X["year_listed"] < year).values, (X["year_listed"] == year).values
    forward[year] = rmse(y[te], make_hgb().fit(X[tr], y[tr]).predict(X[te]))

summary = pd.Series({"Random 5-fold": random_cv.mean(), "Group 5-fold (new neighbourhoods)": group_cv.mean(),
                     "Forward chaining (2021-2024)": np.mean(list(forward.values()))}, name="CV RMSE")
print(summary.round(1).to_string())
print("\nforward-chaining RMSE by test year:", {k: round(v, 1) for k, v in forward.items()})

Random 5-fold                        115.7
Group 5-fold (new neighbourhoods)    171.4
Forward chaining (2021-2024)         137.0

forward-chaining RMSE by test year: {2021: 144.0, 2022: 122.7, 2023: 112.9, 2024: 168.2}


### 🔎 Same model, three "truths"

* **Random K-fold** is the most optimistic (≈ 116): neighbours of every test row sit in the training set.
* **Group K-fold** is far worse (≈ 171): when a whole neighbourhood is unseen, the model can't know its price premium. *If you plan to expand to new areas, random K-fold would have misled you by ~50%.*
* **Forward chaining** lands in between (≈ 137): each year's prices drift upward (≈ 5%/yr), and trees cannot extrapolate (remember Lesson 02!). The 2024 error is the largest: the future is always the hardest test.

> 💡 **Rule:** *Validate the way you will deploy.* Mismatch between validation design and real use is a top cause of "great in the notebook, terrible in production".

## 4 · Data leakage: the silent killer

**Leakage** = information from outside the training fold sneaking into model fitting, inflating validation scores. Famous culprits: scaling/imputing/encoding on *all* data before splitting, **feature selection** on all data, duplicates across folds, target-derived features, future information.

### 🧪 Worked Example 6.1: a model that "predicts" pure noise
We create 100 samples with **2,000 random features** and **random labels**. There is *nothing to learn*; any honest accuracy must be ≈ 50%.

In [7]:
rng = np.random.default_rng(0)
X_noise = rng.normal(size=(100, 2000))
y_noise = rng.integers(0, 2, 100)                                 # labels are coin flips
cv_noise = StratifiedKFold(5, shuffle=True, random_state=0)

# WRONG: pick the 20 "best" features using ALL the data, then cross-validate
X_selected = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
acc_leaky = cross_val_score(LogisticRegression(max_iter=2000), X_selected, y_noise, cv=cv_noise).mean()

# RIGHT: feature selection is a pipeline step, re-done inside every training fold
honest = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=2000))
acc_honest = cross_val_score(honest, X_noise, y_noise, cv=cv_noise).mean()

print(f"leaky CV accuracy : {acc_leaky:.2f}   <- a 'discovery' in pure noise!")
print(f"honest CV accuracy: {acc_honest:.2f}   <- the truth")

leaky CV accuracy : 0.81   <- a 'discovery' in pure noise!
honest CV accuracy: 0.52   <- the truth


The leaky workflow "finds" an 80%-accurate model in random noise. Why? Among 2,000 random features, some correlate with the labels *by chance*; selecting them using **all** the labels bakes the validation labels into the model. Inside a pipeline, selection sees only training-fold labels, and the honest 50% appears.

> ⚠️ **This exact mistake has appeared in published papers.** Habit: *every* data-dependent transformation (imputation, scaling, encoding, selection, target encoding, PCA) lives inside the `Pipeline`.

## 5 · Reading learning curves: do I need more data or a better model?
`learning_curve` retrains on growing fractions of the training data and compares **training** and **validation** scores.

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (name, model) in zip(axes, [("Ridge on log(price)  (simple)", TransformedTargetRegressor(Ridge(1.0), func=np.log, inverse_func=np.exp)),
                                     ("HistGB  (flexible)", HistGradientBoostingRegressor(learning_rate=0.05, max_iter=300, max_depth=4, random_state=0))]):
    sizes, tr_sc, va_sc = learning_curve(housing_pipe(model), X, y, cv=KFold(5, shuffle=True, random_state=SEED),
                                         train_sizes=np.linspace(0.1, 1.0, 6), scoring="neg_root_mean_squared_error")
    for sc, label, color in [(-tr_sc, "training", PALETTE["blue"]), (-va_sc, "validation", PALETTE["orange"])]:
        ax.plot(sizes, sc.mean(axis=1), "o-", label=label, color=color)
        ax.fill_between(sizes, sc.mean(axis=1) - sc.std(axis=1), sc.mean(axis=1) + sc.std(axis=1), alpha=.15, color=color)
    ax.set_title(name); ax.set_xlabel("training rows"); ax.legend()
axes[0].set_ylabel("RMSE")
plt.tight_layout(); plt.show()

How to read these:

* **Big gap between the curves** (training ≪ validation) = **high variance / overfitting**: more data or more regularisation helps. The flexible HistGB shows this gap; the validation curve is still sloping down → *more data would help*.
* **Curves converge but at a high error** = **high bias / underfitting**: more data won't help: you need a richer model or better features. (The simple log-linear model's curves meet at a higher error.)

## 6 · Comparing models *properly*

Now the main event. Suppose we have four candidates and want to know which is genuinely better: not just luckier on one split.

**The protocol**
1. Use the **same folds** for every model (paired comparison).
2. Use **repeated** K-fold (here 5 folds × 3 repeats = **15 scores per model**) to get a distribution of scores rather than a point.
3. Compare **paired differences**, with an honest significance test.
4. Weigh the practical cost: is a 0.5% gain worth 10× the training time?

In [9]:
candidates = {
    "LogLinear":    TransformedTargetRegressor(Ridge(1.0), func=np.log, inverse_func=np.exp),
    "RandomForest": RandomForestRegressor(n_estimators=150, max_features=0.4, random_state=SEED, n_jobs=-1),
    "HistGB":       HistGradientBoostingRegressor(learning_rate=0.05, max_iter=300, max_depth=4, random_state=0),
    "XGBoost":      xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, subsample=0.8, random_state=0),
}
rkf = RepeatedKFold(n_splits=5, n_repeats=3, random_state=SEED)          # fixed seed => IDENTICAL folds for every model

cv_scores, fit_times = {}, {}
for name, model in candidates.items():
    res = cross_validate(housing_pipe(model), X, y, cv=rkf, scoring="neg_root_mean_squared_error", n_jobs=1)
    cv_scores[name], fit_times[name] = -res["test_score"], res["fit_time"].mean()
cv_scores = pd.DataFrame(cv_scores)                       # rows = 15 folds, columns = models

summary = pd.DataFrame({"mean RMSE": cv_scores.mean(), "std over folds": cv_scores.std(ddof=1),
                        "std. error": cv_scores.std(ddof=1) / np.sqrt(len(cv_scores)), "fit s / fold": pd.Series(fit_times)}).sort_values("mean RMSE")
summary.round(2)

,mean RMSE,std over folds,std. error,fit s / fold
XGBoost,109.44,11.48,2.97,0.12
HistGB,115.16,11.39,2.94,0.24
LogLinear,130.36,25.80,6.66,0.02
RandomForest,130.86,13.68,3.53,0.77


In [10]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
order = summary.index
sns.boxplot(data=cv_scores[list(order)], ax=axes[0], color=PALETTE["blue"])
axes[0].set_ylabel("fold RMSE"); axes[0].set_title("Distribution of 15 fold scores per model")

# the same data, but connected by fold: lines that stay parallel = a paired comparison
for i in range(len(cv_scores)):
    axes[1].plot(range(len(order)), cv_scores.loc[i, order], color=PALETTE["gray"], alpha=.35, lw=1)
axes[1].plot(range(len(order)), cv_scores[order].mean(), "o-", color=PALETTE["red"], lw=2.5, label="mean")
axes[1].set_xticks(range(len(order))); axes[1].set_xticklabels(order); axes[1].set_title("Each grey line = one fold, seen by every model"); axes[1].legend()
plt.tight_layout(); plt.show()

The boxplots overlap heavily (some folds are simply hard for everyone), but the grey lines on the right slope the **same way** fold after fold. That parallelism is the visual signature of a reliable paired difference.

### Is the difference *statistically* real?

Take the 15 per-fold differences $d_i = \text{RMSE}^{A}_i - \text{RMSE}^{B}_i$ between two models. The textbook paired t-test is
$t = \bar d / \sqrt{s_d^2/k}$.

**But it is invalid here.** The 15 fold-scores are *not* independent: different folds share most of their training rows, and repeated CV reuses the same data over and over. The naive test therefore **underestimates the variance** and cries "significant!" too easily.

**Nadeau & Bengio's correction (2003)** inflates the variance to account for the overlap:

$$t = \frac{\bar d}{\sqrt{\Big(\dfrac1k + \dfrac{n_{\text{test}}}{n_{\text{train}}}\Big)\,s_d^2}}\qquad\text{with } k-1 \text{ degrees of freedom.}$$

With 5-fold CV, $n_{\text{test}}/n_{\text{train}} = 1/4$: that term dwarfs $1/k = 1/15$, which is why the corrected test is **much** more conservative.

### 🧪 Worked Example 6.2: XGBoost vs HistGB

In [11]:
d = (cv_scores["XGBoost"] - cv_scores["HistGB"]).values          # 15 paired differences; negative = XGBoost better
k, n_train, n_test = len(d), 2400, 600                           # 5-fold on 3000 rows

naive = stats.ttest_1samp(d, 0)
t_corr = d.mean() / np.sqrt((1 / k + n_test / n_train) * d.var(ddof=1))
p_corr = 2 * stats.t.sf(abs(t_corr), df=k - 1)

print(f"mean difference (XGB − HistGB): {d.mean():+.2f} RMSE   (XGBoost better in {(d < 0).sum()} of {k} folds)")
print(f"naive paired t-test     : t = {naive.statistic:6.2f}   p = {naive.pvalue:.5f}")
print(f"corrected resampled test: t = {t_corr:6.2f}   p = {p_corr:.5f}")

mean difference (XGB − HistGB): -5.71 RMSE   (XGBoost better in 14 of 15 folds)
naive paired t-test     : t =  -5.49   p = 0.00008
corrected resampled test: t =  -2.52   p = 0.02461


Notice how the corrected p-value (≈ 0.025) is hundreds of times larger than the naive one (≈ 0.00008): the *same data* look far less conclusive once we account for the overlap. Always report the corrected version (or use a method designed for dependent folds).

### Statistical vs. practical significance
A tiny p-value says the difference is **real**, not that it is **important**. Ask: *does a 5-point RMSE improvement (about 5%) change a decision or earn money? What does the better model cost to train, deploy and maintain?* A p-value never answers those.

### The one-standard-error rule
When several models are statistically close, prefer the **simplest** one:

> Find the best model's mean score and its standard error. Among all models whose mean is within **one standard error** of the best, pick the **simplest / cheapest**.

It builds in the humility that CV scores are noisy and that simpler models are easier to explain, deploy and trust.

## 7 · Nested cross-validation: estimating performance *when you also tune*

If you tune hyperparameters with CV and then report **that same CV score**, the number is optimistic: you picked the configuration that happened to look best on those folds: selection bias, a milder cousin of leakage.

**Nested CV** fixes it with two loops:
* **inner loop:** tuning (e.g. `GridSearchCV`) using only the outer-training data;
* **outer loop:** measures how well the *whole procedure* ("tune, then fit") performs on data the tuning never saw.

The bias is largest with **small data and big search spaces**, so we'll demonstrate on a 250-row sample with 40 tree configurations.

In [12]:
idx = np.random.default_rng(0).choice(len(X), 250, replace=False)
Xs, ys = X.iloc[idx], y.iloc[idx]
grid = {"decisiontreeregressor__max_depth": [2, 4, 6, None], "decisiontreeregressor__min_samples_leaf": [1, 3, 8, 15, 25],
        "decisiontreeregressor__max_features": [0.5, 1.0]}                       # 4 x 5 x 2 = 40 configurations
make_search = lambda cv: GridSearchCV(make_pipeline(make_preprocessor(Xs), DecisionTreeRegressor(random_state=0)), grid, cv=cv,
                                      scoring="neg_root_mean_squared_error")

# non-nested: tune with CV and report the CV score of the winner
plain = make_search(KFold(5, shuffle=True, random_state=1)).fit(Xs, ys)
non_nested = -plain.best_score_

# nested: outer folds measure the whole 'tune-then-fit' procedure on rows the tuner never saw
outer_scores = []
for tr, te in KFold(5, shuffle=True, random_state=2).split(Xs):
    search = make_search(KFold(3, shuffle=True, random_state=1)).fit(Xs.iloc[tr], ys.iloc[tr])
    outer_scores.append(rmse(ys.iloc[te], search.predict(Xs.iloc[te])))
nested = float(np.mean(outer_scores))

print(f"non-nested CV score of the tuned winner : {non_nested:6.1f}   <- what a naive report would say")
print(f"nested CV estimate of the whole pipeline : {nested:6.1f}   <- the honest number")

non-nested CV score of the tuned winner :  251.8   <- what a naive report would say
nested CV estimate of the whole pipeline :  258.2   <- the honest number


The nested estimate is **worse (higher RMSE)** than the naive one (the difference is the optimism from picking the winner of 40 contestants on the same folds. With big data and a small grid the gap shrinks; with small data and a large search (or Bayesian optimisation with hundreds of trials) Lesson 07) it can become the dominant error in your report.

**Practical advice:** for final model selection on small/medium data, use nested CV; on large data, a clean *train / validation / test* triple split is fine: tune on validation, touch the test set **once**.

## 8 · Choosing the metric: it *is* the question you're asking

The same predictions rank differently under different metrics. Let's score four models on the course test split with six metrics:

In [13]:
X_train, X_test, y_train, y_test = get_housing_split()
from sklearn.metrics import mean_squared_log_error, median_absolute_error

rows = []
for name, model in candidates.items():
    pred = housing_pipe(model).fit(X_train, y_train).predict(X_test)
    r = regression_report(y_test, pred)
    rows.append({"model": name, "RMSE": r["RMSE"], "MAE": r["MAE"], "MedAE": median_absolute_error(y_test, pred),
                 "MAPE %": r["MAPE_%"], "RMSLE": np.sqrt(mean_squared_log_error(y_test, np.clip(pred, 1, None))), "R²": r["R2"]})
metric_table = pd.DataFrame(rows).set_index("model")
metric_table.round(3)

,RMSE,MAE,MedAE,MAPE %,RMSLE,R²
model,,,,,,
LogLinear,108.056,64.108,37.782,14.692,0.199,0.908
RandomForest,116.525,75.066,45.514,18.704,0.223,0.893
HistGB,110.350,64.250,36.393,14.040,0.184,0.904
XGBoost,104.343,58.913,30.890,13.008,0.175,0.914


In [14]:
ranks = metric_table.rank(ascending=True)
ranks["R²"] = metric_table["R²"].rank(ascending=False)
print("rank of each model under each metric (1 = best):")
ranks.astype(int)

rank of each model under each metric (1 = best):


,RMSE,MAE,MedAE,MAPE %,RMSLE,R²
model,,,,,,
LogLinear,2,2,3,3,3,2
RandomForest,4,4,4,4,4,4
HistGB,3,3,2,2,2,3
XGBoost,1,1,1,1,1,1


| metric | penalises | use it when |
|---|---|---|
| **RMSE** | big errors heavily | large misses are disproportionately costly (e.g. under-pricing a mansion) |
| **MAE / MedAE** | all errors linearly / typical error | you care about the *typical* miss, robust to outliers |
| **MAPE** | *relative* error | prices span orders of magnitude and "10% off" means the same for every house, but breaks near zero and favours under-prediction |
| **RMSLE** | relative error, asymmetric | multiplicative targets; under-prediction penalised more than over-prediction |
| **R²** | variance explained vs the mean baseline | communicating overall fit (not for choosing between models on different data) |

In this particular split one model (XGBoost) wins on every metric, so the metric didn't change the champion, but look at the **runner-up**: LogLinear is 2nd on RMSE, MAE and R² (it handles the big, expensive houses well), while HistGB is 2nd on MedAE, MAPE and RMSLE (it is better on the *typical* house, in relative terms). On another split, or for a different shortlist, the *winner* could change too. **Which metric is "right" depends on the decision the model supports**: pick it *before* looking at results, not after.

## 9 · A model-selection checklist (print this)

1. Define the **deployment scenario** → choose the splitter (random / stratified / group / time).
2. Choose the **metric** before running anything; always include a **baseline**.
3. Wrap all preprocessing in a **Pipeline**.
4. Evaluate candidates on **identical repeated folds**; look at mean ± SE, not a single number.
5. Test differences with a **corrected** paired test; consider **effect size** and **cost**.
6. Apply the **one-SE rule** for ties.
7. If tuning, **nest** (small data) or hold out a pristine **test set** (large data).
8. Touch the test set **once**; report an interval, not a point.

### ✋ Check your understanding

Try to answer *out loud* before you open the answer. If you can't say it simply, you don't own it yet.

**Q1. Two models score RMSE 108.2 and 107.8 on one 20% test split. Can you conclude the second is better?**

<details><summary>Show answer</summary>

No. A single split's RMSE varies by ±10% or more across random splits, so a 0.4-point difference is meaningless on its own. Compare on identical repeated folds, look at the *paired* differences and use a corrected test.

</details>

**Q2. Why is comparing models fold-by-fold (paired) more powerful than comparing their average scores?**

<details><summary>Show answer</summary>

Hard folds are hard for *every* model. Taking the difference within a fold cancels the shared fold difficulty, leaving only the model-vs-model signal, so the variance of the paired difference is much smaller than the variance of either model's raw score.

</details>

**Q3. Why must feature selection be inside the cross-validation loop?**

<details><summary>Show answer</summary>

Selection uses the labels. If it sees *all* labels, the validation labels influence which features are chosen, so validation scores are inflated: we showed 80% accuracy on pure noise. Inside a pipeline, each training fold selects features using only its own labels.

</details>

**Q4. When would you choose `GroupKFold` over `KFold`?**

<details><summary>Show answer</summary>

When rows are clustered (same patient, store, neighbourhood) and the model will be applied to **new clusters**. Random K-fold puts siblings of every validation row in the training set and overstates performance.

</details>

**Q5. The nested-CV estimate is worse than the naive tuned-CV score. Is the nested one 'wrong'?**

<details><summary>Show answer</summary>

No, it is the *honest* one. The naive score is optimistically biased because the winning configuration was chosen for looking best on those very folds. Nested CV evaluates the entire tune-then-fit procedure on data the tuner never saw.

</details>

## 10 · Problems

Variables available: `cv_scores` (15 fold-RMSEs × 4 models), `summary`, `X`, `y`, `X_noise`, `y_noise`, `cv_noise`, `rmse`, `housing_pipe`, `make_hgb`.

### 🧩 Problem 06.1 ⭐: K-fold from scratch

Write `my_kfold_indices(n, k, seed)` that returns a **list of `k` pairs** `(train_idx, val_idx)` of NumPy index arrays:

* shuffle the row indices with `np.random.default_rng(seed)`,
* split them into `k` nearly equal folds,
* fold *i* is the validation set in pair *i*; the other folds form its training set.

Every index must appear in exactly one validation fold, and the fold sizes may differ by at most 1.

<details><summary>💡 Hint 1</summary>

`rng = np.random.default_rng(seed); order = rng.permutation(n)` gives a shuffled index list.

</details>

<details><summary>💡 Hint 2</summary>

`np.array_split(order, k)` cuts it into k nearly equal pieces.

</details>

<details><summary>💡 Hint 3</summary>

For fold i: `val = folds[i]`; `train = np.concatenate([folds[j] for j in range(k) if j != i])`.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [15]:
def my_kfold_indices(n, k, seed=0):
    order = np.random.default_rng(seed).permutation(n)
    folds = np.array_split(order, k)
    return [(np.concatenate([folds[j] for j in range(k) if j != i]), folds[i]) for i in range(k)]

# ---- self-check ----
def _ok():
    splits = my_kfold_indices(103, 5, seed=3)
    val_all = np.concatenate([v for _, v in splits])
    sizes = [len(v) for _, v in splits]
    disjoint = all(len(np.intersect1d(tr, va)) == 0 and len(tr) + len(va) == 103 for tr, va in splits)
    return len(splits) == 5 and sorted(val_all) == list(range(103)) and max(sizes) - min(sizes) <= 1 and disjoint
_probe = my_kfold_indices(10, 2)
check("06.1", _ok, "Return 5 (train, val) pairs; validation folds must partition 0..102 and train/val must be disjoint.", needs=[_probe])

✅ 06.1: correct -- nice work!


**✅ Solution walk-through (Problem 06.1)**

That's all `KFold(shuffle=True)` does: permute, cut into k pieces, rotate the validation role. Everything else in scikit-learn's splitters is a variation on *how the pieces are chosen* (stratified, grouped, ordered in time).

### 🧩 Problem 06.2 ⭐⭐: Catch the leak

Using the pure-noise data (`X_noise`, `y_noise`, `cv_noise`):

* `acc_leaky`: select the top **10** features with `SelectKBest(f_classif, k=10)` on *all* the data, **then** cross-validate a `LogisticRegression(max_iter=2000)`.
* `acc_clean`: the same, but with selection inside a `Pipeline`.

The leaky accuracy should look impressive and the clean one should hover around 0.5. Then explain *why* in one sentence of your own.

<details><summary>💡 Hint 1</summary>

Leaky: `Xk = SelectKBest(f_classif, k=10).fit_transform(X_noise, y_noise)`, then `cross_val_score(LogisticRegression(max_iter=2000), Xk, y_noise, cv=cv_noise).mean()`.

</details>

<details><summary>💡 Hint 2</summary>

Clean: `make_pipeline(SelectKBest(f_classif, k=10), LogisticRegression(max_iter=2000))` passed to `cross_val_score` with the *raw* `X_noise`.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [16]:
Xk = SelectKBest(f_classif, k=10).fit_transform(X_noise, y_noise)
acc_leaky = cross_val_score(LogisticRegression(max_iter=2000), Xk, y_noise, cv=cv_noise).mean()

pipe_clean = make_pipeline(SelectKBest(f_classif, k=10), LogisticRegression(max_iter=2000))
acc_clean = cross_val_score(pipe_clean, X_noise, y_noise, cv=cv_noise).mean()

# ---- self-check ----
check("06.2", lambda: acc_leaky > 0.65 and 0.35 < acc_clean < 0.62 and acc_leaky - acc_clean > 0.1,
      "The leaky estimate should be well above 0.65; the clean one should be near 0.5.", needs=[acc_leaky, acc_clean])

✅ 06.2: correct -- nice work!


**✅ Solution walk-through (Problem 06.2)**

With selection done on all labels, the 10 chosen features are the ones that happen to correlate with these particular coin flips: *including the validation rows' flips*. The model then 'predicts' validation labels it indirectly saw. Inside a pipeline, each fold selects using only its training labels, and the noise stays noise (≈ 0.5).

### 🧩 Problem 06.3 ⭐⭐: The corrected resampled t-test

Compare **RandomForest** and **HistGB** using the paired fold differences `d = cv_scores["RandomForest"] - cv_scores["HistGB"]`
(15 folds; each fold trains on 2,400 rows and tests on 600).

Compute and store:
* `p_naive`: the p-value of the ordinary paired t-test (`stats.ttest_1samp(d, 0)`),
* `t_corrected` and `p_corrected`: using the Nadeau–Bengio formula (two-sided, `k-1` degrees of freedom),
* `significant`: `True` if `p_corrected < 0.05`.

<details><summary>💡 Hint 1</summary>

`k = len(d)`; the corrected denominator is `np.sqrt((1/k + 600/2400) * d.var(ddof=1))`.

</details>

<details><summary>💡 Hint 2</summary>

`t_corrected = d.mean() / that_denominator`; two-sided p-value: `2 * stats.t.sf(abs(t_corrected), df=k - 1)`.

</details>

<details><summary>💡 Hint 3</summary>

`significant = bool(p_corrected < 0.05)`.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [17]:
d = (cv_scores["RandomForest"] - cv_scores["HistGB"]).values
k = len(d)

p_naive = stats.ttest_1samp(d, 0).pvalue
t_corrected = d.mean() / np.sqrt((1 / k + 600 / 2400) * d.var(ddof=1))
p_corrected = 2 * stats.t.sf(abs(t_corrected), df=k - 1)
significant = bool(p_corrected < 0.05)

# ---- self-check ----
def _ref():
    dd = (cv_scores["RandomForest"] - cv_scores["HistGB"]).values; kk = len(dd)
    t = dd.mean() / np.sqrt((1 / kk + 0.25) * dd.var(ddof=1)); return t, 2 * stats.t.sf(abs(t), df=kk - 1)
check("06.3", lambda: abs(t_corrected - _ref()[0]) < 1e-6 and abs(p_corrected - _ref()[1]) < 1e-9 and p_naive < p_corrected
      and significant == bool(_ref()[1] < 0.05),
      "Use n_test/n_train = 600/2400 = 0.25 and df = k - 1. The corrected p-value must be larger than the naive one.",
      needs=[p_naive, t_corrected, p_corrected, significant])

✅ 06.3: correct -- nice work!


**✅ Solution walk-through (Problem 06.3)**

In our run HistGB beats the random forest by ≈ 16 RMSE points on average, **in all 15 folds**. The naive test gives p ≈ 1e-7; after the overlap correction p ≈ 5e-4: still comfortably significant, but about 4,000× less extreme. Compare with XGBoost vs HistGB in Example 6.2 (≈ 6 points, 14 of 15 folds), where the corrected p-value is only ≈ 0.025. **Same machinery, different strength of evidence: that is the point of testing.**

### 🧩 Problem 06.4 ⭐⭐: Apply the one-standard-error rule

Models are ranked by *simplicity* (1 = simplest): `LogLinear=1, HistGB=2, XGBoost=3, RandomForest=4`.

Using `cv_scores`:
1. find the model with the lowest mean RMSE (`best_model`),
2. compute its standard error (`std(ddof=1)/sqrt(15)`),
3. gather every model whose mean RMSE is within **one standard error** of the best,
4. store the **simplest** of those in `chosen`.

<details><summary>💡 Hint 1</summary>

`means = cv_scores.mean()`; `best_model = means.idxmin()`; `se = cv_scores[best_model].std(ddof=1) / np.sqrt(len(cv_scores))`.

</details>

<details><summary>💡 Hint 2</summary>

`eligible = [m for m in means.index if means[m] <= means[best_model] + se]`.

</details>

<details><summary>💡 Hint 3</summary>

`chosen = min(eligible, key=complexity.get)`.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [18]:
complexity = {"LogLinear": 1, "HistGB": 2, "XGBoost": 3, "RandomForest": 4}

means = cv_scores.mean()
best_model = means.idxmin()
se = cv_scores[best_model].std(ddof=1) / np.sqrt(len(cv_scores))
eligible = [m for m in means.index if means[m] <= means[best_model] + se]
chosen = min(eligible, key=complexity.get)

# ---- self-check ----
def _ref():
    means = cv_scores.mean(); b = means.idxmin(); se = cv_scores[b].std(ddof=1) / np.sqrt(len(cv_scores))
    return b, min([m for m in means.index if means[m] <= means[b] + se], key={"LogLinear": 1, "HistGB": 2, "XGBoost": 3, "RandomForest": 4}.get)
check("06.4", lambda: (best_model, chosen) == _ref(), "Find the best mean, add one standard error, pick the simplest eligible model.",
      needs=[best_model, chosen])

✅ 06.4: correct -- nice work!


**✅ Solution walk-through (Problem 06.4)**

XGBoost has the lowest mean RMSE (≈ 109.4) with a standard error of ≈ 3.0, so the eligibility threshold is ≈ 112.4. HistGB (≈ 115.2) misses it by about 3 points, and LogLinear / RandomForest (≈ 130) are far off, so the only eligible model is XGBoost itself and the rule **keeps the winner**. The rule only swaps in a simpler model when it is genuinely within noise of the best; here the evidence says XGBoost is worth its (small) extra complexity.

### 🧩 Problem 06.5 ⭐⭐⭐: Forward-chaining validation

Write forward-chaining validation for the housing data and compare it with random 5-fold CV **on the same test rows**:

1. For each test year in `[2021, 2022, 2023, 2024]`, train `make_hgb()` on rows with `year_listed < year`, predict rows with `year_listed == year`, and store the RMSE in the dict `time_rmse`.
2. `mean_time_rmse`: the average of those four numbers.
3. Compute out-of-fold predictions with `cross_val_predict(make_hgb(), X, y, cv=KFold(5, shuffle=True, random_state=SEED))`, then compute the RMSE **on the rows from those same four years only** → `random_rmse`.

The time-aware estimate should be the more pessimistic (higher) one.

<details><summary>💡 Hint 1</summary>

Boolean masks: `tr = (X['year_listed'] < year).values`, `te = (X['year_listed'] == year).values`. Remember `y` is a Series: use `y[tr]` / `y[te]` or `.values`.

</details>

<details><summary>💡 Hint 2</summary>

For the random baseline, build one mask `mask = X['year_listed'].isin([2021, 2022, 2023, 2024]).values` and compute `rmse(y[mask], oof[mask])`.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [19]:
time_rmse = {}
for year in [2021, 2022, 2023, 2024]:
    tr, te = (X["year_listed"] < year).values, (X["year_listed"] == year).values
    time_rmse[year] = rmse(y[te], make_hgb().fit(X[tr], y[tr]).predict(X[te]))

mean_time_rmse = float(np.mean(list(time_rmse.values())))

oof = cross_val_predict(make_hgb(), X, y, cv=KFold(5, shuffle=True, random_state=SEED))
mask = X["year_listed"].isin([2021, 2022, 2023, 2024]).values
random_rmse = rmse(y[mask], oof[mask])

# ---- self-check ----
check("06.5", lambda: len(time_rmse) == 4 and abs(mean_time_rmse - np.mean(list(time_rmse.values()))) < 1e-9
      and 100 < random_rmse < 140 and mean_time_rmse > random_rmse,
      "Four yearly RMSEs; the forward-chaining mean (≈ 137) should exceed the random-CV RMSE on the same years (≈ 125).",
      needs=[mean_time_rmse, random_rmse])

✅ 06.5: correct -- nice work!


**✅ Solution walk-through (Problem 06.5)**

Forward chaining is the more pessimistic (≈ 137 vs ≈ 125 on the same rows) because (i) the model has never seen the future's price level (trees can't extrapolate) and (ii) random CV leaks information from neighbouring-in-time rows. Both effects exist in production, so the forward-chaining number is the better forecast of live performance. Mitigations: model a de-trended or log target, add a linear trend component, or retrain frequently.

In [20]:
scoreboard()

Scoreboard: 5 solved, 0 to fix, 0 not attempted
   ✅ 06.1
   ✅ 06.2
   ✅ 06.3
   ✅ 06.4
   ✅ 06.5


## 🏁 Recap

* One train/test split is a **lottery** (±10%+); *paired* differences on identical folds are far more stable.
* **K-fold** rotates the validation role; choose **stratified / repeated / group / time-aware** splitters to match deployment.
* **Leakage** (selection, scaling, imputation on all data) inflates scores: use **Pipelines**. We "found" 80% accuracy in pure noise.
* Compare models with **paired fold scores**, the **corrected resampled t-test**, **effect size**, **cost**, and the **one-standard-error rule**.
* **Nested CV** gives an honest estimate of "tune-then-fit"; the naive tuned-CV score is optimistic.
* The **metric** encodes the business question: decide it first.

### ➡️ Next up: **Lesson 07: Hyperparameter Tuning**
Now that we can evaluate honestly, how do we *search* efficiently? Grid vs random vs successive halving vs **Bayesian optimisation**: with an animation of a surrogate model hunting for the optimum, and Optuna on real boosting models.